# SMS Spam Classification using Improved SVM
Dataset: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

This version tests word-level and character-level TF-IDF features with Linear SVM.
The test set is held out until final evaluation. Run the notebook from top to bottom.


In [ ]:
# Install (if needed)
!pip -q install wordcloud nltk scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)


## Upload the SMS dataset
Upload the Kaggle `spam.csv` file when prompted. The loading cell below detects the uploaded CSV filename.


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load the uploaded CSV file (works with spam.csv or SMS.csv)
import os

csv_candidates = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_candidates:
    raise FileNotFoundError("Please upload the SMS spam dataset CSV file.")
csv_path = csv_candidates[0]

raw_df = pd.read_csv(csv_path, encoding="latin-1")
if not {"v1", "v2"}.issubset(raw_df.columns):
    raise ValueError("Expected Kaggle SMS dataset columns 'v1' and 'v2'.")

df = raw_df[["v1", "v2"]].copy()
df.columns = ["label", "message"]
df = df.dropna(subset=["label", "message"])
df["label"] = df["label"].str.strip().str.lower()
df = df[df["label"].isin(["ham", "spam"])]
df = df.drop_duplicates(subset=["label", "message"]).reset_index(drop=True)

print("Dataset:", csv_path)
print("Shape:", df.shape)
df.head()


In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print("Duplicates:",df.duplicated().sum())
print(df['label'].value_counts())


## Exploratory Data Analysis

In [ ]:
df['char_count']=df['message'].str.len()
df['word_count']=df['message'].str.split().str.len()

fig,ax=plt.subplots(1,2,figsize=(10,4))
sns.countplot(data=df,x='label',ax=ax[0])
sns.histplot(data=df,x='word_count',hue='label',bins=30,ax=ax[1])
plt.tight_layout()
plt.show()




In [ ]:
spam=' '.join(df[df.label=='spam'].message)
ham=' '.join(df[df.label=='ham'].message)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(WordCloud(background_color='white').generate(spam))
plt.axis('off'); plt.title("Spam")
plt.subplot(1,2,2)
plt.imshow(WordCloud(background_color='white').generate(ham))
plt.axis('off'); plt.title("Ham")
plt.show()


## Preprocessing

In [ ]:
## Text preprocessing

For SMS spam, numbers, punctuation, URLs, and unusual character patterns can be useful signals.
So this version uses the original message text rather than deleting those clues with aggressive cleaning.
TF-IDF vectorization is fitted inside the model pipeline to avoid learning vocabulary from the test set.


## Train/Test Split

In [ ]:
## Train/test split

# Map ham -> 0 and spam -> 1
df["target"] = df["label"].map({"ham": 0, "spam": 1}).astype(int)

X = df["message"].astype(str)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("\nTraining label proportions:")
print(y_train.value_counts(normalize=True).rename(index={0: "ham", 1: "spam"}))


## SVM Model

In [ ]:
## Word + character TF-IDF with Linear SVM

from sklearn.pipeline import FeatureUnion
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

features = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=30000,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("char_tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        max_features=30000,
        sublinear_tf=True
    ))
])

pipeline = Pipeline([
    ("features", features),
    ("svm", LinearSVC(max_iter=10000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    "svm__C": [0.1, 0.3, 1, 3, 10],
    "svm__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation spam F1:", round(grid.best_score_, 4))


## Final evaluation on the untouched test set


In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"], digits=4))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Ham", "Spam"])
disp.plot(values_format="d")
plt.title("SMS Spam Classification — Test Set")
plt.show()


## Cross-validation summary


In [ ]:
# This is the CV score from training data only; test data remains separate.
print("Best mean CV spam F1:", round(grid.best_score_, 4))
print("Final test metrics above are the unbiased holdout evaluation.")


## Try a new SMS message


In [ ]:
model = grid.best_estimator_

def predict_sms(text):
    prediction = int(model.predict([str(text)])[0])
    return "Spam" if prediction == 1 else "Ham"

print(predict_sms("Congratulations! You have won a free iPhone. Click here now."))
print(predict_sms("Hi, are we meeting tomorrow at 3 PM?"))
